imports

In [5]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from preprocessing import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Path

In [6]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Data"
PDF_PATH = DATA_DIR / "Os Maias - Eca de Queiros.pdf"
OUTPUT_PATH = DATA_DIR / "novel_text.txt"

Diagnóstico

In [7]:
doc = pymupdf.open(PDF_PATH)
page = doc[10]
prev = None
for l in read_page_lines(page)[:15]:
    gap = "" if prev is None else round(l["y"] - prev, 1)
    print(f'gap={gap!s:>5} x0={l["x0"]:6.1f} | {l["text"][:60]}')
    prev = l["y"]

gap=      x0=  85.1 | bordado a ouro, uma pele de urso branco, e uma venerável cad
gap= 31.6 x0=  85.1 | cuja tapeçaria mostrava ainda as armas dos Maias no desmaio 
gap= 31.4 x0=  85.1 | seda.
gap= 41.5 x0=  85.1 | No corredor do segundo andar guarnecido com retratos de famí
gap= 31.6 x0=  85.1 | os quartos de Afonso. Carlos dispusera os seus, num ângulo d
gap= 31.4 x0=  85.1 | entrada particular, e janelas sobre o jardim: eram três gabi
gap= 31.6 x0=  85.1 | portas, unidos pelo mesmo tapete: e os recostos acolchoados,
gap= 31.4 x0=  85.1 | forrava as paredes, faziam dizer ao Vilaça que aquilo não er
gap= 31.6 x0=  85.1 | médico — mas de dançarina!
gap= 41.5 x0=  85.1 | A casa, depois de arranjada, ficou vazia enquanto Carlos, já
gap= 31.4 x0=  85.1 | uma longa viagem pela Europa; — e foi só nas vésperas da sua
gap= 31.6 x0=  85.1 | lindo Outono de 1875, que Afonso se resolveu enfim a deixar 
gap= 31.4 x0=  85.1 | vir instalar-se no Ramalhete. Havia vinte e cinco anos que e
gap= 31.6 

Executar o pré-processamento

In [8]:
novel_text = preprocess_novel(pdf_path=PDF_PATH, output_path=OUTPUT_PATH)

Reading PDF: 100%|██████████| 1004/1004 [00:02<00:00, 489.03it/s]


line_gap=31.6  para_gap=42  threshold=36.8

Preprocessing completed.
Raw characters: 1277124
Novel characters: 1276409
Novel words: 218715
Saved to: c:\SGI_Final\SGI-Project\Data\novel_text.txt


Verificar

In [9]:
import re, random
paras = novel_text.split("\n\n")
print(len(paras), "parágrafos |", sum(len(p.split()) for p in paras) // len(paras), "palavras/parágrafo")
print(len(re.findall(r"^CAP[IÍ]TULO", novel_text, re.M)), "capítulos")   # creio que são 18
print(sum(p.startswith("—") for p in paras), "parágrafos de diálogo")
for p in random.sample(paras, 4):
    print(repr(p[:160]), "\n")

5728 parágrafos | 38 palavras/parágrafo
18 capítulos
2706 parágrafos de diálogo
'Mas, exclamava Carlos, nunca imaginara ela que fatalmente tudo se descobriria um dia?' 

'Parou diante do grande espelho, deu um olhar descontente ao seu jaquetão claro e às botas com mau verniz.' 

'— Uma coisa assim! Creia o amigo! Foi essa mulher, essa mulher que aí apareceu, que o matou! Nunca foi o mesmo depois daquele abalo! Não foi mais nada! Foi isso' 

'— O Guimarães, o que está em Paris!' 



Tudo


In [10]:
NOVEL_TEXT_PATH = DATA_DIR / "novel_text.txt"
novel_text = NOVEL_TEXT_PATH.read_text(encoding="utf-8")
cleaned_text = clean_extracted_text(novel_text)
print(cleaned_text[:2000])

CAPÍTULO I

A casa que os Maias vieram habitar em Lisboa, no Outono de 1875, era conhecida na vizinhança da Rua de S. Francisco de Paula, e em todo o bairro das Janelas Verdes, pela Casa do Ramalhete, ou simplesmente o Ramalhete. Apesar deste fresco nome de vivenda campestre, o Ramalhete, sombrio casarão de paredes severas, com um renque de estreitas varandas de ferro no primeiro andar, e por cima uma tímida fila de janelinhas abrigadas à beira do telhado, tinha o aspeto tristonho de residência eclesiástica que competia a uma edificação do reinado da senhora D. Maria I: com uma sineta e com uma cruz no topo, assemelhar-se-ia a um colégio de Jesuítas. O nome de Ramalhete provinha decerto de um revestimento quadrado de azulejos fazendo painel no lugar heráldico do Escudo de Armas, que nunca chegara a ser colocado, e representando um grande ramo de girassóis atado por uma fita onde se distinguiam letras e números de uma data.

Longos anos o Ramalhete permanecera desabitado, com teias de a

In [11]:
txt = OUTPUT_PATH.read_text(encoding="utf-8")
print(repr(txt[:400]))
print(len(txt.split("\n\n")), "parágrafos no ficheiro")

'CAPÍTULO I\n\nA casa que os Maias vieram habitar em Lisboa, no Outono de 1875, era conhecida na vizinhança da Rua de S. Francisco de Paula, e em todo o bairro das Janelas Verdes, pela Casa do Ramalhete, ou simplesmente o Ramalhete. Apesar deste fresco nome de vivenda campestre, o Ramalhete, sombrio casarão de paredes severas, com um renque de estreitas varandas de ferro no primeiro andar, e por cima'
5728 parágrafos no ficheiro


recomedaçao do claude para separar os paragrafos para a co occurence

In [12]:
from pathlib import Path
from preprocessing_utils import *

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Data"
NOVEL_TEXT_PATH = DATA_DIR / "novel_text.txt"

novel_text = NOVEL_TEXT_PATH.read_text(encoding="utf-8")
cleaned_text = clean_extracted_text(novel_text)
print(len(cleaned_text.split("\n\n")), "parágrafos em cleaned_text")

5728 parágrafos em cleaned_text


In [13]:
import re
import pandas as pd

def build_paragraph_table(text):
    rows, chapter, para_in_ch = [], 0, 0
    for p in text.split("\n\n"):
        p = p.strip()
        if not p:
            continue
        if re.match(r"^CAP[IÍ]TULO\b", p, re.IGNORECASE):
            chapter += 1
            para_in_ch = 0
            continue
        para_in_ch += 1
        rows.append({
            "paragraph_id": len(rows),
            "chapter": chapter,
            "para_in_chapter": para_in_ch,
            "is_dialogue": p.startswith(("—", "–")),
            "n_words": len(p.split()),
            "text": p,
        })
    return pd.DataFrame(rows)

df = build_paragraph_table(cleaned_text)
df.to_csv(DATA_DIR / "paragraphs.csv", index=False, encoding="utf-8")

print(len(df), "parágrafos (sem os títulos de capítulo)")
print(df.groupby("chapter").size())
df.head(8)

5710 parágrafos (sem os títulos de capítulo)
chapter
1     141
2     160
3     307
4     176
5     254
6     325
7     285
8     258
9     438
10    319
11    316
12    397
13    226
14    418
15    577
16    349
17    510
18    254
dtype: int64


,paragraph_id,chapter,para_in_chapter,is_dialogue,n_words,text
0,0,1,1,False,159,"A casa que os Maias vieram habitar em Lisboa, ..."
1,1,1,2,False,209,Longos anos o Ramalhete permanecera desabitado...
2,2,1,3,False,107,Este inútil pardieiro (como lhe chamava Vilaça...
3,3,1,4,True,20,— Ainda têm um pedaço de pão — disse Vilaça so...
4,4,1,5,False,120,"Os Maias eram uma antiga família da Beira, sem..."
5,5,1,6,False,195,A venda da Tojeira fora realmente aconselhada ...
6,6,1,7,False,49,"Afonso riu muito da frase, e respondeu que aqu..."
7,7,1,8,False,110,"Sua Excelência mandava: — e, como esse Inverno..."


Ver se os paragrafos ficaram bem feitos

In [14]:
odd = df[~df["text"].str[0].str.isupper() & ~df["is_dialogue"]]
print(len(odd), "parágrafos que não começam por maiúscula nem travessão")
print(odd["text"].str[:90].head(15).to_string())

7 parágrafos que não começam por maiúscula nem travessão
238     «É uma fatalidade, parto para sempre com Tancr...
582     «Estão assim dissipadas todas as dúvidas», acr...
1042    retiro rústico. E todavia pareceu a Carlos que...
1329                            «Abril chegou! Sê minha.»
1585                       ambos tão doces como carícias.
4596    «O nosso amigo e distinto sportman, Dâmaso Sal...
5466                            «Vai casar!», pensou Ega.
